# Chicago Crimes — Normalización de CSVs para GCP

## Schema definido

| Columna | Tipo BigQuery | Notas |
|---|---|---|
| unique_key | INT64 | PK |
| case_number | STRING | |
| date | TIMESTAMP | UTC |
| block | STRING | |
| iucr | STRING | Contiene letras |
| primary_type | STRING | |
| description | STRING | |
| location_description | STRING | |
| arrest | BOOL | |
| domestic | BOOL | |
| beat | INT64 | |
| district | INT64 | Nullable |
| ward | INT64 | Nullable |
| community_area | INT64 | Nullable |
| fbi_code | STRING | Contiene letras |
| x_coordinate | INT64 | Nullable, Illinois State Plane |
| y_coordinate | INT64 | Nullable |
| year | INT64 | |
| updated_on | TIMESTAMP | UTC |
| latitude | FLOAT64 | Nullable |
| longitude | FLOAT64 | Nullable |
| location | STRING | Formato WKT: POINT (lon lat) |

In [1]:
import pandas as pd
import numpy as np
import os
import re

FOLDER = 'Chicago_Crimes_by_Year'

# Mapeo de columnas Formato B/C (Title Case) → snake_case
COLUMN_MAP = {
    'ID': 'unique_key',
    'Case Number': 'case_number',
    'Date': 'date',
    'Block': 'block',
    'IUCR': 'iucr',
    'Primary Type': 'primary_type',
    'Description': 'description',
    'Location Description': 'location_description',
    'Arrest': 'arrest',
    'Domestic': 'domestic',
    'Beat': 'beat',
    'District': 'district',
    'Ward': 'ward',
    'Community Area': 'community_area',
    'FBI Code': 'fbi_code',
    'X Coordinate': 'x_coordinate',
    'Y Coordinate': 'y_coordinate',
    'Year': 'year',
    'Updated On': 'updated_on',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'Location': 'location'
}

EXPECTED_COLUMNS = list(COLUMN_MAP.values())
print('Schema definido con', len(EXPECTED_COLUMNS), 'columnas')
print(EXPECTED_COLUMNS)

Schema definido con 22 columnas
['unique_key', 'case_number', 'date', 'block', 'iucr', 'primary_type', 'description', 'location_description', 'arrest', 'domestic', 'beat', 'district', 'ward', 'community_area', 'fbi_code', 'x_coordinate', 'y_coordinate', 'year', 'updated_on', 'latitude', 'longitude', 'location']


In [2]:
# ── Funciones de normalización ──────────────────────────────────────────────

def parse_date(series):
    """Detecta el formato de fecha y convierte a UTC."""
    sample = series.dropna().iloc[0] if not series.dropna().empty else ''
    # Formato A: '2001-10-03 17:00:00+00:00'
    if '+00:00' in str(sample) or (len(str(sample)) > 4 and str(sample)[4] == '-'):
        return pd.to_datetime(series, utc=True, errors='coerce')
    # Formato B/C: '12/31/2014 11:58:00 PM'
    return pd.to_datetime(series, format='%m/%d/%Y %I:%M:%S %p', errors='coerce').dt.tz_localize('UTC')

def parse_updated_on(series):
    """Detecta el formato de updated_on y convierte a UTC."""
    sample = series.dropna().iloc[0] if not series.dropna().empty else ''
    # Formato A: '2015-08-17 15:03:40+00:00'
    if '+00:00' in str(sample) or (len(str(sample)) > 4 and str(sample)[4] == '-'):
        return pd.to_datetime(series, utc=True, errors='coerce')
    # Formato B/C: '2018 Feb 10 03:50:01 PM'
    return pd.to_datetime(series, format='%Y %b %d %I:%M:%S %p', errors='coerce').dt.tz_localize('UTC')

def parse_bool(series):
    """Normaliza True/False / true/false a boolean."""
    return series.astype(str).str.strip().str.lower().map({'true': True, 'false': False}).astype('boolean')

def to_nullable_int(series):
    """Convierte a entero nullable (Int64), tolerando NaN y floats como 1.0."""
    return pd.to_numeric(series, errors='coerce').astype('Int64')

def normalize_location(series, is_format_a):
    """Estandariza location a WKT: POINT (lon lat)."""
    if is_format_a:
        # Formato A: '(41.883500187, -87.627876698)' → POINT (lon lat)
        def a_to_wkt(val):
            if pd.isna(val) or str(val).strip() == '':
                return None
            m = re.match(r'\(([^,]+),\s*([^)]+)\)', str(val))
            if m:
                lat, lon = m.group(1).strip(), m.group(2).strip()
                return f'POINT ({lon} {lat})'
            return None
        return series.apply(a_to_wkt)
    else:
        # Formato B/C: ya está en WKT POINT, solo limpiar nulls
        return series.where(series.astype(str).str.strip() != '', other=None)

print('Funciones de normalización definidas.')

Funciones de normalización definidas.


In [3]:
# ── Normalización de todos los CSVs ─────────────────────────────────────────

files = sorted(f for f in os.listdir(FOLDER) if f.endswith('.csv'))
errors = []

for filename in files:
    path = os.path.join(FOLDER, filename)

    # Leer todo como string para no perder ceros a la izquierda ni malinterpretar tipos
    df = pd.read_csv(path, dtype=str, keep_default_na=False)

    # Detectar formato por nombre de primera columna
    is_format_a = df.columns[0] == 'unique_key'

    # 1. Renombrar columnas si es Formato B/C
    if not is_format_a:
        df = df.rename(columns=COLUMN_MAP)

    # Verificar que tenemos todas las columnas esperadas
    missing = [c for c in EXPECTED_COLUMNS if c not in df.columns]
    if missing:
        errors.append(f'{filename}: columnas faltantes {missing}')
        continue

    # Reordenar columnas al orden estándar
    df = df[EXPECTED_COLUMNS]

    # 2. Reemplazar strings vacíos por NaN
    df = df.replace('', np.nan)

    # 3. Normalizar tipos de datos
    df['unique_key']    = to_nullable_int(df['unique_key'])
    df['date']          = parse_date(df['date'])
    df['arrest']        = parse_bool(df['arrest'])
    df['domestic']      = parse_bool(df['domestic'])
    df['beat']          = to_nullable_int(df['beat'])
    df['district']      = to_nullable_int(df['district'])
    df['ward']          = to_nullable_int(df['ward'])
    df['community_area']= to_nullable_int(df['community_area'])
    df['x_coordinate']  = to_nullable_int(df['x_coordinate'])
    df['y_coordinate']  = to_nullable_int(df['y_coordinate'])
    df['year']          = to_nullable_int(df['year'])
    df['updated_on']    = parse_updated_on(df['updated_on'])
    df['latitude']      = pd.to_numeric(df['latitude'], errors='coerce')
    df['longitude']     = pd.to_numeric(df['longitude'], errors='coerce')
    df['location']      = normalize_location(df['location'], is_format_a)

    # 4. Guardar sobreescribiendo el archivo
    df.to_csv(path, index=False)
    print(f'✓ {filename:<40} {len(df):>8,} filas  |  Formato {"A" if is_format_a else "B/C"}')

if errors:
    print('\n⚠ Errores encontrados:')
    for e in errors:
        print(' -', e)
else:
    print(f'\n✓ {len(files)} archivos normalizados sin errores')

C:\Users\Usuario\AppData\Local\Temp\ipykernel_13560\722570708.py:29: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace('', np.nan)


✓ Chicago_Crimes_2001.csv                   485,929 filas  |  Formato A


✓ Chicago_Crimes_2002.csv                   486,826 filas  |  Formato A


✓ Chicago_Crimes_2003.csv                   475,990 filas  |  Formato A


✓ Chicago_Crimes_2004.csv                   469,433 filas  |  Formato A


...

✓ Chicago_Crimes_2023.csv                   262,756 filas  |  Formato A


✓ Chicago_Crimes_2024.csv                   256,305 filas  |  Formato A


✓ Chicago_Crimes_2025.csv                    92,460 filas  |  Formato A


C:\Users\Usuario\AppData\Local\Temp\ipykernel_13560\722570708.py:29: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace('', np.nan)


✓ Chicago_Crimes_2026.csv                    58,429 filas  |  Formato A

✓ 26 archivos normalizados sin errores


In [4]:
# ── Validación post-normalización ───────────────────────────────────────────

print('Validando consistencia de todos los archivos...\n')
print(f'{"Archivo":<40} {"Columnas":>8} {"Filas":>10} {"Formato date"}')
print('─' * 80)

inconsistencias = []

for filename in sorted(f for f in os.listdir(FOLDER) if f.endswith('.csv')):
    path = os.path.join(FOLDER, filename)
    df = pd.read_csv(path, nrows=2)  # Solo header + 1 fila para validar
    ncols = len(df.columns)
    cols_ok = list(df.columns) == EXPECTED_COLUMNS

    df_full = pd.read_csv(path, usecols=['date'], nrows=1)
    sample_date = df_full['date'].iloc[0] if not df_full.empty else 'N/A'

    status = '✓' if cols_ok else '✗'
    print(f'{status} {filename:<38} {ncols:>8} {"":>10} {sample_date}')

    if not cols_ok:
        inconsistencias.append(filename)

if inconsistencias:
    print('\n⚠ Archivos con columnas inconsistentes:', inconsistencias)
else:
    print('\n✓ Todos los archivos tienen el mismo esquema. Listos para GCP.')

Validando consistencia de todos los archivos...

Archivo                                  Columnas      Filas Formato date
────────────────────────────────────────────────────────────────────────────────
✓ Chicago_Crimes_2001.csv                      22            2001-10-03 17:00:00+00:00


✓ Chicago_Crimes_2002.csv                      22            2002-02-24 12:00:00+00:00
✓ Chicago_Crimes_2003.csv                      22            2003-07-11 16:00:00+00:00
✓ Chicago_Crimes_2004.csv                      22            2004-01-28 18:00:00+00:00
✓ Chicago_Crimes_2005.csv                      22            2005-03-18 22:00:00+00:00
✓ Chicago_Crimes_2006.csv                      22            2006-06-02 08:00:00+00:00
✓ Chicago_Crimes_2007.csv                      22            2007-10-23 22:35:00+00:00
✓ Chicago_Crimes_2008.csv                      22            2008-09-17 02:20:00+00:00


✓ Chicago_Crimes_2009.csv                      22            2009-09-16 13:00:00+00:00
✓ Chicago_Crimes_2010.csv                      22            2010-03-02 08:46:00+00:00
✓ Chicago_Crimes_2011.csv                      22            2011-07-15 17:30:00+00:00
✓ Chicago_Crimes_2012.csv                      22            2012-08-03 14:15:00+00:00
✓ Chicago_Crimes_2013.csv                      22            2013-09-23 22:00:00+00:00
✓ Chicago_Crimes_2014.csv                      22            2014-12-31 23:58:00+00:00
✓ Chicago_Crimes_2015.csv                      22            2015-04-23 17:00:00+00:00
✓ Chicago_Crimes_2016.csv                      22            2016-10-04 13:00:00+00:00
✓ Chicago_Crimes_2017.csv                      22            2017-11-20 17:05:00+00:00


✓ Chicago_Crimes_2018.csv                      22            2018-07-14 17:35:00+00:00
✓ Chicago_Crimes_2019.csv                      22            2019-05-28 17:59:00+00:00
✓ Chicago_Crimes_2020.csv                      22            2020-03-29 03:45:00+00:00
✓ Chicago_Crimes_2021.csv                      22            2021-05-02 19:30:00+00:00
✓ Chicago_Crimes_2022.csv                      22            2022-11-02 06:40:00+00:00
✓ Chicago_Crimes_2023.csv                      22            2023-04-06 02:00:00+00:00
✓ Chicago_Crimes_2024.csv                      22            2024-05-18 14:56:00+00:00
✓ Chicago_Crimes_2025.csv                      22            2025-01-01 00:00:00+00:00
✓ Chicago_Crimes_2026.csv                      22            2026-04-12 00:00:00+00:00

✓ Todos los archivos tienen el mismo esquema. Listos para GCP.
